**Geochemistry Biplot App for Bruker Results.csv Files**

Voila App Version
N. Tripcevich 2026, CC BY-SA 4.0  
[More Information Online](https://github.com/arf-berkeley/bruker-xrf-ppm-plot)


In [7]:
#%% capture
%pip install plotly ipywidgets voila
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display, HTML
import ipywidgets as widgets
from io import StringIO
import base64

# Shared state dictionary — used by all tabs
state = {
    'study_import':          None,
    'study_import_filtered': None,
    'study':                 None,
}

# Columns that are never element data
NON_ELEMENT = [
    'File #', 'DateTime', 'Name', 'Application',
    'Method', 'ElapsedTime', 'Elapsed', '_batch', '_method'
]

Note: you may need to restart the kernel to use updated packages.


**Select the Results.csv table from your Bruker analysis**

Browse to a copy of the __Results.csv__ file typically found in Bruker/Data/Results.csv

Importing all of the Weight Percent data from the most recent Method used.

In [8]:
# Cell 2
# # ── parser ────────────────────────────────────────────────────────────────
def parse_results_csv(content_str):
    content_str = content_str.replace('\r\n', '\n').replace('\r', '\n')
    lines = [l for l in content_str.split('\n') if l.strip()]

    segment_starts = [
        i for i, line in enumerate(lines)
        if line.split(',')[0].strip().strip('"') == 'File #'
    ]
    if not segment_starts:
        raise ValueError('No "File #" header row found.')

    frames = []
    for idx, start in enumerate(segment_starts, start=1):
        end           = segment_starts[idx] if idx < len(segment_starts) else len(lines)
        segment_lines = lines[start:end]
        if len(segment_lines) < 2:
            continue
        try:
            df_seg = pd.read_csv(
                StringIO('\n'.join(segment_lines)),
                dtype=str,
                skipinitialspace=True
            )
        except Exception:
            continue
        df_seg.dropna(how='all', inplace=True)
        df_seg.dropna(axis=1, how='all', inplace=True)
        if df_seg.empty:
            continue
        df_seg['_batch'] = idx
        frames.append(df_seg)

    if not frames:
        raise ValueError('No data found after parsing all segments.')

    df = pd.concat(frames, ignore_index=True, join='outer')
    df.replace({'< LOD': None, 'None': None, '': None}, inplace=True)
    return df


# ── cleaner ────────────────────────────────────────────────────────────────
def clean_data(df):
    DROP = [
        'Alloy 1', 'Match Qual 1', 'Alloy 2', 'Match Qual 2',
        'Alloy 3', 'Match Qual 3', 'Multiplier', 'Cal Check',
        'Operator', 'Field1', 'Field2', 'ID', '_batch', '_method'
    ]
    drop_cols = [c for c in df.columns if 'Err' in c or c in DROP]
    out       = df[[c for c in df.columns if c not in drop_cols]].copy()
    out       = out.replace('< LOD', 0)

    str_cols  = [c for c in ['Name', 'Application', 'Method'] if c in out.columns]
    num_cols  = [c for c in out.columns if c not in str_cols + ['DateTime']]
    out[str_cols] = out[str_cols].astype('string')
    out[num_cols] = out[num_cols].apply(pd.to_numeric, errors='coerce')
    out['DateTime'] = pd.to_datetime(out['DateTime'], errors='coerce')

    el_cols      = [c for c in num_cols if c not in ['File #', 'ElapsedTime']]
    out[el_cols] = (out[el_cols] * 10000).round(1)
    out[el_cols] = out[el_cols].fillna(0)

    out.dropna(axis=1, how='all', inplace=True)
    out.dropna(how='all', inplace=True)
    return out


# ── helpers ────────────────────────────────────────────────────────────────
def _app_str(df):
    return df['Application'].astype(object).fillna('').astype(str).str.strip()

def get_elements(df):
    return [
        c for c in df.columns
        if c not in NON_ELEMENT
        and pd.api.types.is_numeric_dtype(df[c])
    ]

def prep_plot_df(df):
    out = df.copy()
    for col in out.columns:
        if str(out[col].dtype) in ('string', 'StringDtype') or out[col].dtype == object:
            out[col] = out[col].astype(object).fillna('(no name)').astype(str)
    if 'Name' in out.columns:
        out['Name'] = out['Name'].replace(
            {'nan': '(no name)', 'None': '(no name)', '<NA>': '(no name)'}
        )
    return out

def get_unique_apps():
    df = state['study_import']
    if df is None:
        return []
    return sorted(
        df['Application'].dropna().astype(str).str.strip()
        .replace('', pd.NA).dropna().unique().tolist()
    )

def batches_for(application):
    df = state['study_import'].copy()
    if application != 'All applications':
        df = df[_app_str(df) == application.strip()]
    return ['All'] + [str(b) for b in sorted(df['_batch'].dropna().unique())]

print('Cell 2 ready.')

Cell 2 ready.


Cleaning data includes removing the following: elemental error columns, Alloy, Match Qual columns, Multiplier, Cal Check, Operator, Field 1&2. This script also replaces Below Detection Limits LOD with 0.

You may now select rows of data from recent analyses by filtering with either File # or Date.

In [9]:
##Cell 3 
# # ── Tab 1 : Load & Filter ──────────────────────────────────────────────────

# widgets
upload_widget = widgets.FileUpload(
    accept='.csv',
    multiple=False,
    layout=widgets.Layout(width='300px')
)
load_btn = widgets.Button(
    description='Load File',
    button_style='success',
    icon='upload',
    disabled=True,
    layout=widgets.Layout(width='150px')
)
load_status    = widgets.Label('Upload a Bruker XRF Results.csv file')
load_out       = widgets.Output()
app_dd         = widgets.Dropdown(
    options=[],
    description='Application:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='360px')
)
batch_dd       = widgets.Dropdown(
    options=[],
    description='Batch:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='180px')
)
apply_btn      = widgets.Button(
    description='Apply Filter',
    button_style='primary',
    icon='filter',
    disabled=True,
    layout=widgets.Layout(width='150px')
)
filter_summary = widgets.Output()
filter_result  = widgets.Output()

# callbacks
def on_upload_change(change):
    load_btn.disabled = not bool(upload_widget.value)
    if upload_widget.value:
        load_status.value = 'File ready — click Load File'

def on_load(btn):
    with load_out:
        load_out.clear_output(wait=True)
        try:
            raw     = upload_widget.value[0]['content'].tobytes()
            content = raw.decode('utf-8', errors='replace')
            df      = parse_results_csv(content)
            state['study_import'] = df
            apps             = ['All applications'] + get_unique_apps()
            default_app      = apps[1] if len(apps) > 1 else 'All applications'
            app_dd.options   = apps
            app_dd.value     = default_app
            batch_dd.options = batches_for(default_app)
            batch_dd.value   = 'All'
            apply_btn.disabled   = False
            load_btn.disabled    = True
            load_btn.description = 'Loaded ✓'
            load_status.value    = f'✓ Loaded {df.shape[0]} rows'
            refresh_filter_summary(default_app)
        except Exception as e:
            load_status.value = f'Error: {e}'

def refresh_filter_summary(application):
    with filter_summary:
        filter_summary.clear_output(wait=True)
        df = state['study_import'].copy()
        if application != 'All applications':
            df = df[_app_str(df) == application.strip()]
        batches = sorted(df['_batch'].unique().tolist())
        dates   = pd.to_datetime(df['DateTime'], errors='coerce').dropna()
        d_min   = dates.min().strftime('%m-%d-%Y') if not dates.empty else '?'
        d_max   = dates.max().strftime('%m-%d-%Y') if not dates.empty else '?'
        print(f'  "{application}" → {len(df)} rows')
        print(f'  Batch(es) : {batches}')
        print(f'  Date range: {d_min} – {d_max}')

def on_app_change(change):
    if state['study_import'] is None:
        return
    batch_dd.options = batches_for(change['new'])
    batch_dd.value   = 'All'
    refresh_filter_summary(change['new'])

def on_apply(btn):
    with filter_result:
        filter_result.clear_output(wait=True)
        df        = state['study_import'].copy()
        sel_app   = app_dd.value
        sel_batch = batch_dd.value

        if sel_app != 'All applications':
            df = df[_app_str(df) == sel_app.strip()]
        if sel_batch != 'All':
            df = df[df['_batch'] == int(sel_batch)]

        df = df.reset_index(drop=True)
        state['study_import_filtered'] = df

        if df.empty:
            print(f'No rows for application="{sel_app}" batch="{sel_batch}"')
            return

        cleaned        = clean_data(df)
        state['study'] = cleaned

        dates = pd.to_datetime(df['DateTime'], errors='coerce').dropna()
        d_min = dates.min().strftime('%m-%d-%Y %H:%M') if not dates.empty else '?'
        d_max = dates.max().strftime('%m-%d-%Y %H:%M') if not dates.empty else '?'

        try:
            file_nums    = df['File #'].dropna().astype(int)
            f_min, f_max = int(file_nums.min()), int(file_nums.max())
        except Exception:
            f_min = f_max = '?'

        print(f'✓ {len(cleaned)} rows ready')
        print(f'  Application : {sel_app}')
        print(f'  Batch       : {sel_batch}')
        print(f'  File # range: {f_min} – {f_max}')
        print(f'  Date range  : {d_min} – {d_max}')
        print(f'  Elements    : {get_elements(cleaned)}')
        print('\n→ Switch tabs to view data, plots and export.')

        render_table()
        render_biplot()
        render_ternary()
        render_export()

upload_widget.observe(on_upload_change, names='value')
load_btn.on_click(on_load)
app_dd.observe(on_app_change, names='value')
apply_btn.on_click(on_apply)

tab1 = widgets.VBox([
    widgets.HTML('<h3>📂 Load &amp; Filter</h3>'),
    widgets.HTML('<span style="color:grey;font-size:12px">Step 1 — Upload your Bruker XRF Results.csv file</span>'),
    widgets.HTML('<hr>'),
    upload_widget,
    widgets.HBox([load_btn, load_status]),
    load_out,
    widgets.HTML('<hr>'),
    widgets.HTML('<span style="color:grey;font-size:12px">Step 2 — Filter by Application and Batch then click Apply Filter</span>'),
    widgets.HBox([app_dd, batch_dd, apply_btn]),
    filter_summary,
    filter_result,
], layout=widgets.Layout(padding='10px'))

print('Cell 3 ready.')

Cell 3 ready.


In [10]:
filter_output = widgets.Output()
results_output = widgets.Output()

def show_filters():
    global study
    study = clean_data(study_import)[0]
    
    with filter_output:
        filter_output.clear_output(wait=True)
        
        min_file = int(study['File #'].min())
        max_file = int(study['File #'].max())
        min_date = study['DateTime'].min().date()
        max_date = study['DateTime'].max().date()

        file_slider = widgets.IntRangeSlider(
            value=[min_file, max_file],
            min=min_file,
            max=max_file,
            step=1,
            description='File #:',
            continuous_update=False,
            layout=widgets.Layout(width='500px')
        )

        start_picker = widgets.DatePicker(
            description='Start Date:',
            value=min_date,
            min=min_date,
            max=max_date
        )
        end_picker = widgets.DatePicker(
            description='End Date:',
            value=max_date,
            min=min_date,
            max=max_date
        )

        apply_btn = widgets.Button(
            description='Apply Filter',
            button_style='primary',
            icon='filter'
        )
        filter_status = widgets.Label(
            f'Data available: File # {min_file} to {max_file} | '
            f'{min_date} to {max_date}'
        )

        def apply_filter(btn):
            global study
            file_min, file_max = file_slider.value
            start = pd.Timestamp(start_picker.value)
            end = pd.Timestamp(end_picker.value) + pd.Timedelta(days=1)

            study = clean_data(study_import)[0]
            study = study[
                (study['File #'] >= file_min) &
                (study['File #'] <= file_max) &
                (study['DateTime'] >= start) &
                (study['DateTime'] < end)
            ].copy()

            filter_status.value = (
                f'✓ {len(study)} rows selected | '
                f'File # {file_min} to {file_max} | '
                f'{start.date()} to {end_picker.value}'
            )
            show_results()

        apply_btn.on_click(apply_filter)

        display(widgets.VBox([
            widgets.HTML('<b>Filter Data</b>'),
            file_slider,
            widgets.HBox([start_picker, end_picker]),
            apply_btn,
            filter_status
        ]))

display(filter_output)
display(results_output)

Output()

Output()

In [11]:
# Cell 4
# ── Tab 2 : Data Table ─────────────────────────────────────────────────────

table_out = widgets.Output()

def get_clean_element_list(df_columns):
    # This list covers the most common XRF elements
    valid_elements = [
        'Al', 'Si', 'P', 'S', 'Cl', 'K', 'Ca', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 
        'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'As', 'Se', 'Br', 'Rb', 'Sr', 'Y', 
        'Zr', 'Nb', 'Mo', 'Ag', 'Cd', 'Sn', 'Sb', 'Ba', 'La', 'Ce', 'Pr', 
        'Nd', 'W', 'Au', 'Hg', 'Pb', 'Th', 'U'
    ]
    
    found_elements = []
    for col in df_columns:
        # Clean the column name: remove (ppm), Err, and extra spaces
        clean_name = col.split(' ')[0].split('(')[0].strip()
        
        # Only keep it if it's a real element and NOT an error/uncertainty column
        if clean_name in valid_elements and 'Err' not in col and '+/-' not in col:
            found_elements.append(col)
            
    return sorted(list(set(found_elements)))

def render_table():
    with table_out:
        table_out.clear_output(wait=True)
        df = state['study']
        if df is None:
            print('No data yet — upload and filter in the Load & Filter tab first.')
            return

        display_df = df.copy()
        display_df['DateTime'] = display_df['DateTime'].dt.strftime('%m/%d/%Y %H:%M')
        display_df = display_df.rename(columns={'ElapsedTime': 'Elapsed'})

        non_element = ['File #', 'DateTime', 'Name', 'Application', 'Method', 'Elapsed']
        text_cols   = [c for c in ['DateTime', 'Name', 'Application', 'Method']
                       if c in display_df.columns]
        el_cols     = [c for c in display_df.columns
                       if c not in non_element
                       and pd.api.types.is_numeric_dtype(display_df[c])]

        display(
            display_df.style
            .format({col: '{:.1f}' for col in el_cols})
            .set_properties(**{
                'text-align': 'right',
                'font-size':  '12px'
            })
            .set_properties(subset=text_cols, **{
                'text-align': 'left'
            })
            .set_table_styles([{
                'selector': 'th',
                'props': [('text-align', 'center'), ('font-weight', 'bold')]
            }])
            .hide(axis='index')
        )

tab2 = widgets.VBox([
    widgets.HTML('<h3>📋 Data Table</h3>'),
    widgets.HTML('<span style="color:grey;font-size:12px">Cleaned data in PPM. Apply a filter first to populate this table.</span>'),
    widgets.HTML('<hr>'),
    table_out,
], layout=widgets.Layout(padding='10px'))

print('Cell 4 ready.')

Cell 4 ready.


In [12]:
# Cell 5
# ── Tab 3 : Biplot ─────────────────────────────────────────────────────────

# 1. Get the list of actual elements from your uploaded file
el_options = get_clean_element_list(df.columns)

# 2. Safety Check: If no elements found, use all columns as a fallback
if not el_options:
    el_options = list(df.columns)

# 3. Choose defaults safely
# We try to find Sr/Rb, but if they aren't there, we just take the 1st and 2nd element found
default_x = 'Sr' if 'Sr' in el_options else el_options[0]
default_y = 'Rb' if 'Rb' in el_options else (el_options[1] if len(el_options) > 1 else el_options[0])

# 4. Create the Dropdowns using these safe defaults
x_axis = widgets.Dropdown(
    options=el_options,
    value=default_x,
    description='X-axis:',
)

y_axis = widgets.Dropdown(
    options=el_options,
    value=default_y,
    description='Y-axis:',
)
log_x = widgets.Checkbox(value=False, description='Log X',
                         layout=widgets.Layout(width='90px'))
log_y = widgets.Checkbox(value=False, description='Log Y',
                         layout=widgets.Layout(width='90px'))

biplot_out = widgets.Output()

def render_biplot():
    df = state['study']
    if df is None:
        return
    els = get_elements(df)
    if len(els) < 2:
        return
    x_dd.unobserve_all()
    y_dd.unobserve_all()
    log_x.unobserve_all()
    log_y.unobserve_all()
    # Reset value to None before changing options
    x_dd.value   = None
    y_dd.value   = None
    x_dd.options = els
    y_dd.options = els
    x_dd.value   = 'Sr' if 'Sr' in els else els[0]
    y_dd.value   = 'Rb' if 'Rb' in els else els[1]
    x_dd.observe(on_biplot_change, names='value')
    y_dd.observe(on_biplot_change, names='value')
    log_x.observe(on_biplot_change, names='value')
    log_y.observe(on_biplot_change, names='value')
    draw_biplot()

def draw_biplot():
    with biplot_out:
        biplot_out.clear_output(wait=True)
        df = state['study']
        if df is None:
            print('No data yet — upload and filter in the Load & Filter tab first.')
            return

        x = x_dd.value
        y = y_dd.value

        if not x or not y:
            print('Please select X and Y elements.')
            return

        plot_df    = prep_plot_df(df)
        color_col  = 'Name' if 'Name' in plot_df.columns else None
        name_order = sorted(plot_df['Name'].unique().tolist()) if color_col else []
        hover_data = [c for c in ['File #', 'DateTime']
                      if c in plot_df.columns and c not in [x, y]]

        try:
            fig = px.scatter(
                plot_df,
                x=x,
                y=y,
                color=color_col,
                category_orders={'Name': name_order},
                hover_data=hover_data,
                title=f'{y} vs {x}  (n={len(plot_df)})',
                labels={
                    x: f'{x} (PPM)',
                    y: f'{y} (PPM)',
                    'Name': 'Sample Name'
                }
            )
            fig.update_traces(marker=dict(size=8, opacity=0.85))
            fig.update_xaxes(type='log' if log_x.value else 'linear')
            fig.update_yaxes(type='log' if log_y.value else 'linear')
            fig.update_layout(
                height=600,
                hovermode='closest',
                legend=dict(
                    title=dict(text='Sample Name', font=dict(size=13)),
                    itemsizing='constant',
                    bordercolor='lightgrey',
                    borderwidth=1,
                    bgcolor='rgba(255,255,255,0.85)',
                    x=1.02, xanchor='left',
                    y=1,    yanchor='top'
                ),
                margin=dict(r=180)
            )
            fig.show()
        except Exception as e:
            print(f'Plot error: {e}')

def on_biplot_change(change):
    draw_biplot()

tab3 = widgets.VBox([
    widgets.HTML('<h3>📊 Biplot</h3>'),
    widgets.HTML('<span style="color:grey;font-size:12px">Select two elements to plot. Apply a filter first to populate the dropdowns.</span>'),
    widgets.HTML('<hr>'),
    widgets.HBox([x_dd, y_dd, log_x, log_y]),
    biplot_out,
], layout=widgets.Layout(padding='10px'))

print('Cell 5 ready.')

NameError: name 'df' is not defined

In [13]:
# Cell 6
# ── Tab 4 : Ternary ────────────────────────────────────────────────────────

a_dd = widgets.Dropdown(
    options=[],
    description='A (top):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)
b_dd = widgets.Dropdown(
    options=[],
    description='B (bottom left):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px')
)
c_dd = widgets.Dropdown(
    options=[],
    description='C (bottom right):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='220px')
)

ternary_out = widgets.Output()

def render_ternary():
    df = state['study']
    if df is None:
        return
    els = get_elements(df)
    if len(els) < 3:
        return
    a_dd.unobserve_all()
    b_dd.unobserve_all()
    c_dd.unobserve_all()
    # Reset value to None before changing options
    a_dd.value   = None
    b_dd.value   = None
    c_dd.value   = None
    a_dd.options = els
    b_dd.options = els
    c_dd.options = els
    a_dd.value   = 'Rb' if 'Rb' in els else els[0]
    b_dd.value   = 'Sr' if 'Sr' in els else els[1]
    c_dd.value   = 'Zr' if 'Zr' in els else els[2]
    a_dd.observe(on_ternary_change, names='value')
    b_dd.observe(on_ternary_change, names='value')
    c_dd.observe(on_ternary_change, names='value')
    draw_ternary()
def draw_ternary():
    with ternary_out:
        ternary_out.clear_output(wait=True)
        df = state['study']
        if df is None:
            print('No data yet — upload and filter in the Load & Filter tab first.')
            return

        a = a_dd.value
        b = b_dd.value
        c = c_dd.value

        if not a or not b or not c:
            print('Please select three elements.')
            return

        if len({a, b, c}) < 3:
            print('Please select three different elements.')
            return

        plot_df = prep_plot_df(df)
        plot_df = plot_df.dropna(subset=[a, b, c])

        if plot_df.empty:
            print(f'No rows with valid data for {a}, {b}, {c}.')
            return

        color_col  = 'Name' if 'Name' in plot_df.columns else None
        name_order = sorted(plot_df['Name'].unique().tolist()) if color_col else []
        hover_data = [c for c in ['File #', 'DateTime']
                      if c in plot_df.columns]

        try:
            fig = px.scatter_ternary(
                plot_df,
                a=a,
                b=b,
                c=c,
                color=color_col,
                category_orders={'Name': name_order},
                hover_data=hover_data,
                title=f'Ternary: {a} / {b} / {c}  (n={len(plot_df)})',
                labels={
                    'Name': 'Sample Name',
                    a: f'{a} (PPM)',
                    b: f'{b} (PPM)',
                    c: f'{c} (PPM)'
                }
            )
            fig.update_traces(marker=dict(size=8, opacity=0.85))
            fig.update_layout(
                height=650,
                legend=dict(
                    title=dict(text='Sample Name', font=dict(size=13)),
                    itemsizing='constant',
                    bordercolor='lightgrey',
                    borderwidth=1,
                    bgcolor='rgba(255,255,255,0.85)',
                    x=1.02, xanchor='left',
                    y=1,    yanchor='top'
                ),
                margin=dict(r=180)
            )
            fig.show()
        except Exception as e:
            print(f'Plot error: {e}')

def on_ternary_change(change):
    draw_ternary()

tab4 = widgets.VBox([
    widgets.HTML('<h3>🔺 Ternary Plot</h3>'),
    widgets.HTML('<span style="color:grey;font-size:12px">Select three elements to plot. Apply a filter first to populate the dropdowns.</span>'),
    widgets.HTML('<hr>'),
    widgets.HBox([a_dd, b_dd, c_dd]),
    ternary_out,
], layout=widgets.Layout(padding='10px'))

print('Cell 6 ready.')

Cell 6 ready.


In [14]:
# Cell 7
# ── Tab 5 : Export ─────────────────────────────────────────────────────────

export_btn = widgets.Button(
    description='Export CSV',
    button_style='success',
    icon='download',
    layout=widgets.Layout(width='150px')
)
export_out = widgets.Output()

def render_export():
    with export_out:
        export_out.clear_output(wait=True)
        df = state['study']
        if df is None:
            print('No data yet — upload and filter in the Load & Filter tab first.')
            return
        dates = pd.to_datetime(df['DateTime'], errors='coerce').dropna()
        if not dates.empty:
            date_str = dates.max().strftime('%Y%m%d')
        else:
            date_str = 'export'
        print(f'Ready to export {len(df)} rows.')
        print(f'Filename: Bruker_Results_export_{date_str}.csv')

def on_export(btn):
    with export_out:
        export_out.clear_output(wait=True)
        df = state['study']
        if df is None:
            print('No data to export.')
            return
        try:
            dates = pd.to_datetime(df['DateTime'], errors='coerce').dropna()
            date_str = dates.max().strftime('%Y%m%d') if not dates.empty else 'export'
        except Exception:
            date_str = 'export'

        filename = f'Bruker_Results_export_{date_str}.csv'
        csv_str  = df.to_csv(index=False)
        b64      = base64.b64encode(csv_str.encode()).decode()
        html     = (
            f'<a download="{filename}" '
            f'href="data:text/csv;base64,{b64}" '
            f'style="font-size:14px;font-weight:bold">'
            f'⬇ Click here to download {filename}</a>'
        )
        display(HTML(html))

export_btn.on_click(on_export)

tab5 = widgets.VBox([
    widgets.HTML('<h3>💾 Export</h3>'),
    widgets.HTML('<span style="color:grey;font-size:12px">Export the cleaned PPM data as a CSV file.</span>'),
    widgets.HTML('<hr>'),
    export_btn,
    export_out,
], layout=widgets.Layout(padding='10px'))

print('Cell 7 ready.')

Cell 7 ready.


In [15]:
# Cell 8 
# ── Assemble tabs and launch ────────────────────────────────────────────────

tabs = widgets.Tab(children=[tab1, tab2, tab3, tab4, tab5])
tabs.set_title(0, '📂 Load & Filter')
tabs.set_title(1, '📋 Data Table')
tabs.set_title(2, '📊 Biplot')
tabs.set_title(3, '🔺 Ternary')
tabs.set_title(4, '💾 Export')

tabs.layout = widgets.Layout(
    width='100%',
    min_height='800px'
)

display(widgets.VBox([
    widgets.HTML('''
        <div style="
            background: #2c3e50;
            color: white;
            padding: 12px 20px;
            border-radius: 6px;
            margin-bottom: 10px;
        ">
            <h2 style="margin:0;padding:0">
                Bruker XRF Geochemistry Viewer
            </h2>
            <span style="font-size:12px;opacity:0.8">
                N. Tripcevich 2026 — CC BY-SA 4.0 &nbsp;|&nbsp;
                Upload a Results.csv file in the Load &amp; Filter tab to begin
            </span>
        </div>
    '''),
    tabs,
]))

# Prime the placeholder messages in each tab
render_table()
render_export()

NameError: name 'tab3' is not defined